In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import json
import random
import numpy as np
from dotenv.main import load_dotenv

load_dotenv()
REPO_PATH = os.getenv('REPO_PATH')

sys.path.append(REPO_PATH)

def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

1.	Create Test-set of ½ entity answers and ½ num/date answers.
2.	Eval functions - to extract question and context alone in 2 modes - 1 answer and both answers and mix the order of the paragraphs.
3.	LLM as a judge evaluator.
4.	Multiple querries to check self-consistency and see answer percentage.
5.	Call model to get answer (function).

In [10]:
# get dataset:
dataset_qa = load_json(f"{REPO_PATH}\data_files\knowledge_maze_qa.json")
dataset_paragraphs = load_json(f"{REPO_PATH}\data_files\knowledge_maze_paragraphs.json")

In [22]:
def get_context(sample, paragraphs_dict):
    """
    Extracts all paragraph texts relevant to a given sample.

    Args:
        sample (dict): The sample dictionary as described.
        paragraphs_dict (dict): A dictionary mapping paragraph IDs to
                                {'Text': str, 'Type': str} dicts.

    Returns:
        list: A list of text strings from the relevant paragraphs.
    """
    paragraph_ids = set()

    # Collect IDs from Reasoning path 1
    if 'Reasoning path 1' in sample and sample['Reasoning path 1']:
        for step in sample['Reasoning path 1']:
            if 'Paragraph' in step:
                paragraph_ids.add(step['Paragraph'])

    # Collect IDs from Reasoning path 2
    if 'Reasoning path 2' in sample and sample['Reasoning path 2']:
        for step in sample['Reasoning path 2']:
            if 'Paragraph' in step:
                paragraph_ids.add(step['Paragraph'])

    # Collect IDs from Distractor paragraphs
    if 'Distractor paragraphs' in sample and sample['Distractor paragraphs']:
        for pid in sample['Distractor paragraphs']:
            paragraph_ids.add(pid)

    # Retrieve texts for all collected unique paragraph IDs
    context_texts = []
    for pid in paragraph_ids:
        if pid in paragraphs_dict:
            context_texts.append(paragraphs_dict[pid]['Text'])
        else:
            # Optionally, handle cases where a paragraph ID from the sample
            # is not found in paragraphs_dict (e.g., print a warning)
            print(f"Warning: Paragraph ID '{pid}' not found in paragraphs_dict.")

    return context_texts

In [23]:
get_context(dataset_qa[0], dataset_paragraphs)

[]

In [24]:
def find_samples_with_known_paragraphs(dataset, paragraphs_dict):
    """
    Identifies samples from a dataset that reference at least one paragraph
    present in the paragraphs_dict.

    Args:
        dataset (list): A list of sample dictionaries.
        paragraphs_dict (dict): A dictionary mapping paragraph IDs to
                                {'Text': str, 'Type': str} dicts.

    Returns:
        list: A list of 'ID' strings for samples that have at least one
              paragraph ID found in paragraphs_dict.
    """
    samples_with_data_ids = []
    paragraph_keys_set = set(paragraphs_dict.keys()) # For efficient lookups

    for sample in dataset:
        if 'ID' not in sample: # Skip samples without an ID
            continue

        referenced_paragraph_ids = set()

        # Collect IDs from Reasoning path 1
        if 'Reasoning path 1' in sample and sample['Reasoning path 1']:
            for step in sample['Reasoning path 1']:
                if 'Paragraph' in step and step['Paragraph']: # Check if key exists and value is not None/empty
                    referenced_paragraph_ids.add(step['Paragraph'])

        # Collect IDs from Reasoning path 2
        if 'Reasoning path 2' in sample and sample['Reasoning path 2']:
            for step in sample['Reasoning path 2']:
                if 'Paragraph' in step and step['Paragraph']:
                    referenced_paragraph_ids.add(step['Paragraph'])

        # Collect IDs from Distractor paragraphs
        if 'Distractor paragraphs' in sample and sample['Distractor paragraphs']:
            for pid in sample['Distractor paragraphs']:
                if pid: # Check if pid is not None/empty
                    referenced_paragraph_ids.add(pid)

        # Check if any of the sample's paragraph IDs are in our known paragraphs
        # The intersection of two sets gives common elements.
        # If the intersection is not empty, it means there's at least one match.
        if referenced_paragraph_ids.intersection(paragraph_keys_set):
            samples_with_data_ids.append(sample['ID'])
        # Alternative using any():
        # if any(pid in paragraph_keys_set for pid in referenced_paragraph_ids):
        #     samples_with_data_ids.append(sample['ID'])

    return samples_with_data_ids

In [27]:
def get_sample(dataset, sid):
    for sample in dataset:
        if sample['ID'] == sid:
            return sample

In [28]:
sample = get_sample(dataset_qa, find_samples_with_known_paragraphs(dataset_qa, dataset_paragraphs)[0])

In [29]:
get_context(sample, dataset_paragraphs)

['"Lady Bird" is a coming-of-age film that takes place in San Francisco, California. The city serves as a vibrant backdrop for the story, which follows the life of a high school senior named Christine "Lady Bird" McPherson as she navigates the challenges of adolescence, family relationships, and her aspirations for the future. San Francisco\'s unique neighborhoods and cultural landmarks are intricately woven into the narrative, highlighting the protagonist\'s desire for independence and her complex feelings about her hometown. The film captures the essence of the city, reflecting both its beauty and the struggles of its residents.',
 "Los Angeles, known for its Mediterranean climate, ethnic diversity, and the entertainment industry, is often mistakenly referred to as the capital city of California. In reality, it is the largest city in the state and serves as a major cultural and economic hub. The city's sprawling metropolitan area is home to iconic landmarks such as Hollywood, the Get